# ParkVision AI — Intelligent Urban Parking Analytics
Slot-level occupancy **detection** using YOLOv8, built for the CRS Artificial Intelligence
Summative Assessment (UrbanFlow AI / ParkSmart AI scenario).

Your dataset (a Roboflow export of PKLot) already has **bounding-box labels in COCO JSON format**,
split into `train/`, `valid/`, `test/` folders, each with its own `_annotations.coco.json` sitting
next to the images. This notebook converts those to YOLO format, trains a real object detector
(so the app draws actual bounding boxes on parking slots — no guessing needed), evaluates it, and
exports the trained weights for the Streamlit app.

Run cells top to bottom. Runtime → Change runtime type → **T4 GPU** before you start.

## 1. Mount Google Drive and locate the dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!ls /content/drive/MyDrive

## 1b. Confirm GPU is active

**Do this before anything else.** Training on CPU instead of GPU can be 20-50x slower (hours instead
of minutes). If this prints `False`, go to Runtime → Change runtime type → set Hardware accelerator
to **T4 GPU** → Save, then re-run from the top (the restart wipes `/content`, so Drive mount and
extraction need to run again, but nothing else is lost).

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only — fix this before training!")

Edit `DRIVE_DATASET_PATH` below to match what you uploaded.
- If you uploaded a `.zip`, point this at the zip file and keep `IS_ZIP = True`.
- If you uploaded an already-extracted folder, point this at that folder and set `IS_ZIP = False`.

In [ ]:
DRIVE_DATASET_PATH = "/content/drive/MyDrive/pklot-dataset.zip"  # <-- EDIT THIS
IS_ZIP = True

import os
EXTRACT_DIR = "/content/pklot_data"

if IS_ZIP:
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    !unzip -q "{DRIVE_DATASET_PATH}" -d {EXTRACT_DIR}
else:
    EXTRACT_DIR = DRIVE_DATASET_PATH

print("Extracted/located at:", EXTRACT_DIR)

In [ ]:
# Confirm the expected Roboflow COCO export structure: train/, valid/, test/,
# each containing images + a single _annotations.coco.json
for split in ["train", "valid", "test"]:
    split_dir = os.path.join(EXTRACT_DIR, split)
    ann_path = os.path.join(split_dir, "_annotations.coco.json")
    exists = os.path.exists(ann_path)
    n_images = len([f for f in os.listdir(split_dir) if f.lower().endswith((".jpg", ".jpeg", ".png"))]) if os.path.isdir(split_dir) else 0
    print(f"{split}: dir exists={os.path.isdir(split_dir)}, annotations found={exists}, image count={n_images}")

## 2. Inspect the COCO annotations

Roboflow PKLot exports typically label two classes (something like `space-empty` / `space-occupied`,
or `free` / `occupied` — naming varies by export). This cell reads the actual category names from
your JSON so nothing downstream is guessed.

In [ ]:
import json

with open(os.path.join(EXTRACT_DIR, "train", "_annotations.coco.json")) as f:
    train_coco = json.load(f)

categories_sorted = sorted(train_coco["categories"], key=lambda c: c["id"])
CLASS_NAMES = [c["name"] for c in categories_sorted]
print("Classes found:", CLASS_NAMES)
print("Number of training images:", len(train_coco["images"]))
print("Number of training annotations (bounding boxes):", len(train_coco["annotations"]))

# Identify which class index means "occupied" vs "empty" for later insight logic.
# Adjust these two lines if your class names differ from what was printed above.
EMPTY_CLASS_NAMES = [n for n in CLASS_NAMES if "empty" in n.lower() or "free" in n.lower() or "vacant" in n.lower()]
OCCUPIED_CLASS_NAMES = [n for n in CLASS_NAMES if "occup" in n.lower() or "busy" in n.lower() or "full" in n.lower()]
print("Detected 'empty' classes:", EMPTY_CLASS_NAMES)
print("Detected 'occupied' classes:", OCCUPIED_CLASS_NAMES)
assert EMPTY_CLASS_NAMES and OCCUPIED_CLASS_NAMES, "Couldn't auto-detect empty/occupied classes — check CLASS_NAMES above and set them manually."


## 3. Convert COCO annotations to YOLO label format

YOLO expects one `.txt` label file per image (same name, `.txt` extension) with lines
`class_id center_x center_y width height`, all normalized 0–1. This writes those files
directly next to each image in `train/`, `valid/`, `test/`.

In [ ]:
def coco_to_yolo_labels(split_dir, class_name_to_id):
    ann_path = os.path.join(split_dir, "_annotations.coco.json")
    with open(ann_path) as f:
        coco = json.load(f)

    cat_id_to_class_id = {c["id"]: class_name_to_id[c["name"]] for c in coco["categories"]}
    images_by_id = {img["id"]: img for img in coco["images"]}

    anns_by_image = {}
    for ann in coco["annotations"]:
        anns_by_image.setdefault(ann["image_id"], []).append(ann)

    written = 0
    for img_id, img in images_by_id.items():
        w, h = img["width"], img["height"]
        lines = []
        for ann in anns_by_image.get(img_id, []):
            x, y, bw, bh = ann["bbox"]  # COCO: top-left x, y, width, height (absolute pixels)
            cx = (x + bw / 2) / w
            cy = (y + bh / 2) / h
            nw = bw / w
            nh = bh / h
            cls_id = cat_id_to_class_id[ann["category_id"]]
            lines.append(f"{cls_id} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}")
        label_path = os.path.join(split_dir, os.path.splitext(img["file_name"])[0] + ".txt")
        with open(label_path, "w") as lf:
            lf.write("\n".join(lines))
        written += 1
    return written

class_name_to_id = {name: i for i, name in enumerate(CLASS_NAMES)}

for split in ["train", "valid", "test"]:
    split_dir = os.path.join(EXTRACT_DIR, split)
    if os.path.isdir(split_dir):
        n = coco_to_yolo_labels(split_dir, class_name_to_id)
        print(f"{split}: wrote {n} label files")

## 4. Write the YOLO data config (`data.yaml`)

In [ ]:
data_yaml_content = f"""path: {EXTRACT_DIR}
train: train
val: valid
test: test

names:
{chr(10).join(f"  {i}: {name}" for i, name in enumerate(CLASS_NAMES))}
"""

DATA_YAML_PATH = os.path.join(EXTRACT_DIR, "data.yaml")
with open(DATA_YAML_PATH, "w") as f:
    f.write(data_yaml_content)

print(data_yaml_content)

## 5. Install Ultralytics (YOLOv8)

In [ ]:
!pip install ultralytics -q

## 6. Train YOLOv8

Defaults below are tuned to finish in roughly 20-40 minutes on a Colab T4 GPU (confirmed active in
step 1b). If GPU is active and you have time for better accuracy, raise `epochs` to 30 and `imgsz`
to 640. If it's still slow, first double-check step 1b actually printed `CUDA available: True` —
that's by far the most common cause of multi-hour "training."


In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")  # nano model: fastest to train, good baseline accuracy

EPOCHS = 15    # raise to 30 for better accuracy if you have time and GPU is confirmed active
IMG_SIZE = 416 # raise to 640 for better accuracy at the cost of slower training
BATCH = 32     # lower to 16 if you hit a CUDA out-of-memory error

results = model.train(
    data=DATA_YAML_PATH,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    project="/content/runs",
    name="parkvision_yolo",
    patience=5,  # early stopping if val performance plateaus
)

## 7. Evaluate on the validation/test set (precision, recall, mAP)

In [ ]:
metrics = model.val(data=DATA_YAML_PATH, split="test")
print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)
print("Precision (per class):", metrics.box.p)
print("Recall (per class):", metrics.box.r)

The training run's `results.png`, `confusion_matrix.png`, and PR curves are saved automatically
under `/content/runs/parkvision_yolo/` — grab these for your README's metrics section and screenshots.

In [ ]:
import shutil
RUN_DIR = "/content/runs/parkvision_yolo"
print("Files saved in this run:")
for f in os.listdir(RUN_DIR):
    print(" -", f)

# Copy key artifacts somewhere easy to find/download
shutil.copy(os.path.join(RUN_DIR, "results.png"), "/content/training_curves.png")
shutil.copy(os.path.join(RUN_DIR, "confusion_matrix.png"), "/content/confusion_matrix.png")
print("Copied results.png -> /content/training_curves.png")
print("Copied confusion_matrix.png -> /content/confusion_matrix.png")

## 8. Save the trained weights (to Colab disk and to Drive)

In [ ]:
BEST_WEIGHTS = os.path.join(RUN_DIR, "weights", "best.pt")

MODEL_PATH_DRIVE = "/content/drive/MyDrive/parkvision_yolo_best.pt"
shutil.copy(BEST_WEIGHTS, MODEL_PATH_DRIVE)

print("Best weights:", BEST_WEIGHTS)
print("Copied to Drive:", MODEL_PATH_DRIVE)
print("Class order used by the model:", CLASS_NAMES)

## 9. Quick sanity check: run inference on one test image

In [ ]:
import glob

test_images = glob.glob(os.path.join(EXTRACT_DIR, "test", "*.jpg")) + glob.glob(os.path.join(EXTRACT_DIR, "test", "*.png"))
assert test_images, "No test images found — check the test/ folder."

sample_result = model.predict(source=test_images[0], conf=0.25, save=True)
print("Detections on sample image:")
for box in sample_result[0].boxes:
    cls_name = CLASS_NAMES[int(box.cls[0])]
    conf = float(box.conf[0])
    print(f"  {cls_name}: {conf:.2f}")

## 10. Parking insight logic

Converts raw YOLO detections into the counts, occupancy %, congestion level, and recommendation
the assignment brief asks for. This same function is reused inside the Streamlit app.

In [ ]:
def parking_insights(detected_class_names, empty_names=EMPTY_CLASS_NAMES, occupied_names=OCCUPIED_CLASS_NAMES):
    """
    detected_class_names: list of class-name strings, one per detected box
    (e.g. ["space-empty", "space-occupied", "space-occupied", ...])
    """
    total = len(detected_class_names)
    occupied = sum(1 for c in detected_class_names if c in occupied_names)
    available = sum(1 for c in detected_class_names if c in empty_names)
    occupancy_pct = (occupied / total * 100) if total > 0 else 0

    if occupancy_pct < 40:
        congestion = "Low"
    elif occupancy_pct <= 75:
        congestion = "Moderate"
    else:
        congestion = "High"

    recommendation = (
        "Parking nearly full — try another location."
        if occupancy_pct >= 90
        else "Slots available — proceed to park."
    )

    return {
        "total_slots": total,
        "occupied_slots": occupied,
        "available_slots": available,
        "occupancy_pct": round(occupancy_pct, 1),
        "congestion_level": congestion,
        "recommendation": recommendation,
    }

# Test with the sample image's real detections
detected_names = [CLASS_NAMES[int(b.cls[0])] for b in sample_result[0].boxes]
print(parking_insights(detected_names))

## 11. Next step

Download `parkvision_yolo_best.pt` from Drive into your local Streamlit app project folder as
`model/parkvision_yolo_best.pt`, then use `app.py` (provided separately) to build the upload → detect
→ annotate → insights dashboard, and push the whole project to GitHub.